## Database Setup and Connection

In [1]:
import os
import re
import html
import torch
import pandas as pd
import plotly.express as px
from transformers import pipeline
from collections import Counter
from datetime import datetime
from dotenv import load_dotenv
from googleapiclient.discovery import build
from sqlalchemy import create_engine, Column, String, Integer, Float, ForeignKey, DateTime, Text, Index, text
from sqlalchemy.orm import declarative_base, relationship, sessionmaker

c:\Users\Priyavarshini\OneDrive\Desktop\GUVI\Final Project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1. Load credentials from .env
load_dotenv()
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
YOUTUBE_API_KEY = os.getenv("YOUTUBE_API_KEY")

In [23]:
# 2. Create the Connection String
DATABASE_URL = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(DATABASE_URL)
Base = declarative_base()
# Initialize YouTube API
youtube = build('youtube', 'v3', developerKey=YOUTUBE_API_KEY)

#### DATABASE SCHEMA DEFINITION

In [4]:
class Channel(Base):
    __tablename__ = 'channels'
    channel_id = Column(String, primary_key=True)
    channel_title = Column(String, nullable=False)
    description = Column(Text)
    subscriber_count = Column(Integer)
    videos = relationship("Video", back_populates="channel")

class Video(Base):
    __tablename__ = 'videos'
    video_id = Column(String, primary_key=True)
    channel_id = Column(String, ForeignKey('channels.channel_id', ondelete="CASCADE"))
    video_title = Column(String, nullable=False)
    published_at = Column(DateTime)
    view_count = Column(Integer)
    like_count = Column(Integer)
    comment_count = Column(Integer)
    channel = relationship("Channel", back_populates="videos")
    comments = relationship("Comment", back_populates="video")

class Comment(Base):
    __tablename__ = 'comments'
    comment_id = Column(String, primary_key=True)
    video_id = Column(String, ForeignKey('videos.video_id', ondelete="CASCADE"))
    author_name = Column(String)
    comment_text = Column(Text, nullable=False)
    like_count = Column(Integer, default=0)
    published_at = Column(DateTime)
    sentiment_label = Column(String)
    sentiment_score = Column(Float)
    video = relationship("Video", back_populates="comments")

# Database Optimization Index
Index('idx_video_id', Comment.video_id)

# Execute Table Creation (Drops old test data, creates fresh tables)
Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)
print("Database Schema Initialized.")

Database Schema Initialized.


#### YouTube API Functions

In [5]:
def parse_yt_duration(duration_str):
    """
    Converts YouTube's ISO 8601 duration (e.g., PT5M30S) to total seconds.
    """
    match = re.match('PT(\d+H)?(\d+M)?(\d+S)?', duration_str)
    if not match: return 0
    hours = int(match.group(1)[:-1]) if match.group(1) else 0
    minutes = int(match.group(2)[:-1]) if match.group(2) else 0
    seconds = int(match.group(3)[:-1]) if match.group(3) else 0
    return hours * 3600 + minutes * 60 + seconds

def get_channel_details(handle):
    username = handle.replace('@', '')
    request = youtube.search().list(q=username, type='channel', part='id,snippet', maxResults=1)
    response = request.execute()
    if not response['items']: return None
    item = response['items'][0]
    return {
        'channel_id': item['id']['channelId'],
        'channel_title': item['snippet']['title'],
        'description': item['snippet']['description']
    }

def get_latest_video_ids(channel_id, limit=10):
    """
    Fetches the latest videos but actively filters OUT YouTube Shorts (<= 60 seconds).
    """
    # 1. Get the Uploads Playlist ID
    ch_request = youtube.channels().list(part='contentDetails', id=channel_id)
    ch_response = ch_request.execute()
    uploads_playlist_id = ch_response['items'][0]['contentDetails']['relatedPlaylists']['uploads']
    
    # 2. Fetch a larger batch (50) so we have enough leftovers after filtering Shorts
    playlist_request = youtube.playlistItems().list(
        part='contentDetails', 
        playlistId=uploads_playlist_id, 
        maxResults=50
    )
    playlist_response = playlist_request.execute()
    raw_video_ids = [item['contentDetails']['videoId'] for item in playlist_response['items']]
    
    # 3. Ask YouTube for the duration of these 50 videos
    long_video_ids = []
    stats_request = youtube.videos().list(
        part='contentDetails',
        id=','.join(raw_video_ids)
    )
    stats_response = stats_request.execute()
    
    # 4. Filter them one by one
    for item in stats_response['items']:
        duration_sec = parse_yt_duration(item['contentDetails']['duration'])
        
        # If it's longer than 60 seconds, it's a standard video!
        if duration_sec > 60:
            long_video_ids.append(item['id'])
            
        # Stop asking once we hit our perfect 10
        if len(long_video_ids) == limit:
            break
            
    return long_video_ids

def get_video_stats(video_ids):
    stats_request = youtube.videos().list(part='snippet,statistics', id=','.join(video_ids))
    stats_response = stats_request.execute()
    video_details = []
    for item in stats_response['items']:
        video_details.append({
            'video_id': item['id'],
            'video_title': item['snippet']['title'],
            'published_at': item['snippet']['publishedAt'],
            'view_count': int(item['statistics'].get('viewCount', 0)),
            'like_count': int(item['statistics'].get('likeCount', 0)),
            'comment_count': int(item['statistics'].get('commentCount', 0))
        })
    return video_details

def get_top_comments_strict(video_id, limit=20):
    comments = []
    try:
        request = youtube.commentThreads().list(part="snippet", videoId=video_id, maxResults=limit, textFormat="plainText")
        response = request.execute()
        for item in response.get('items', [])[:limit]:
            comment = item['snippet']['topLevelComment']['snippet']
            comments.append({
                'comment_id': item['id'],
                'author_name': comment['authorDisplayName'],
                'comment_text': comment['textDisplay'],
                'like_count': comment.get('likeCount', 0),
                'published_at': comment['publishedAt']
            })
    except Exception:
        pass 
    return comments

#### SYNC FUNCTION

In [6]:
def run_10x20_sync(handle):
    print(f"\nStarting 10x20 Sync for: {handle}")
    Session = sessionmaker(bind=engine)
    session = Session()
    
    try:
        ch_info = get_channel_details(handle)
        if not ch_info:
            print("  -> [ERROR] Channel not found.")
            return

        channel_obj = session.query(Channel).filter_by(channel_id=ch_info['channel_id']).first()
        if not channel_obj:
            channel_obj = Channel(**ch_info)
            session.add(channel_obj)
        
        print("  -> Fetching latest 10 videos...")
        v_ids = get_latest_video_ids(ch_info['channel_id'], limit=10)
        v_stats = get_video_stats(v_ids)
        
        total_comments_saved = 0
        
        for v in v_stats:
            video_obj = session.query(Video).filter_by(video_id=v['video_id']).first()
            if not video_obj:
                v['published_at'] = datetime.strptime(v['published_at'], '%Y-%m-%dT%H:%M:%SZ')
                video_obj = Video(channel_id=ch_info['channel_id'], **v)
                session.add(video_obj)
                
            print(f"\n  Video: {v['video_title'][:50]}...")
            comments_data = get_top_comments_strict(v['video_id'], limit=20)
            
            saved_for_this_video = 0
            for c in comments_data:
                existing_c = session.query(Comment).filter_by(comment_id=c['comment_id']).first()
                if not existing_c:
                    c['published_at'] = datetime.strptime(c['published_at'], '%Y-%m-%dT%H:%M:%SZ')
                    comment_obj = Comment(video_id=v['video_id'], **c)
                    session.add(comment_obj)
                    saved_for_this_video += 1
                    total_comments_saved += 1
            
            print(f"Saved {saved_for_this_video} new comments.")

        session.commit()
        print(f"\nSync Complete! Total new comments added to DB: {total_comments_saved}")

    except Exception as e:
        session.rollback()
        print(f"\nA database error occurred: {e}")
    finally:
        session.close()

#### Execution

In [7]:
run_10x20_sync("@mkbhd")


Starting 10x20 Sync for: @mkbhd
  -> Fetching latest 10 videos...

  Video: My Take on The New Apple...
Saved 20 new comments.

  Video: Glass is glass...
Saved 20 new comments.

  Video: So This is Peak Smartphone...
Saved 20 new comments.

  Video: The Unreleased Rollable Smartphone!...
Saved 20 new comments.

  Video: Bluey Phone Review...
Saved 20 new comments.

  Video: So This is Peak Foldable...
Saved 20 new comments.

  Video: The Windows Laptop Problem...
Saved 20 new comments.

  Video: Nothing Phone 4A/Pro Review: I Have a Theory...
Saved 20 new comments.

  Video: Reviewing Everything on my Desk! (2026)...
Saved 20 new comments.

  Video: Macbook Neo Review: Better than you Think!...
Saved 20 new comments.

Sync Complete! Total new comments added to DB: 200


In [8]:
load_dotenv()
engine = create_engine(f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}")

In [9]:
df_videos = pd.read_sql("SELECT * FROM videos", engine)
df_comments = pd.read_sql("SELECT * FROM comments", engine)

print(f"DATASET SHAPE:")
print(f"Total Videos: {len(df_videos)}")
print(f"Total Comments: {len(df_comments)}")

DATASET SHAPE:
Total Videos: 10
Total Comments: 200


In [10]:
# Chart 1: Which video got the most comments in our sample?
comment_counts = df_comments['video_id'].value_counts().reset_index()
comment_counts.columns = ['video_id', 'comment_count']
df_v_merged = pd.merge(comment_counts, df_videos[['video_id', 'video_title']], on='video_id')

fig1 = px.bar(df_v_merged, x='comment_count', y='video_title', orientation='h', 
              title="Comments Extracted per Video", color='comment_count')
fig1.update_layout(yaxis={'categoryorder':'total ascending'})
fig1.show()

In [11]:
# ==========================================
# 3. LENGTH DISTRIBUTION (AI SAFETY CHECK)
# ==========================================
# Calculate character and word lengths
df_comments['char_length'] = df_comments['comment_text'].apply(len)
df_comments['word_count'] = df_comments['comment_text'].apply(lambda x: len(str(x).split()))

# Chart 2: Word Count Distribution
# (Transformers limit is ~512 tokens, which is roughly 350-400 words)
fig2 = px.histogram(df_comments, x='word_count', nbins=50, 
                    title="Comment Length Distribution (Word Count)",
                    color_discrete_sequence=['indianred'])
fig2.add_vline(x=400, line_dash="dash", line_color="black", annotation_text="AI Danger Zone (>400 words)")
fig2.show()

In [12]:
# ==========================================
# 4. WORD FREQUENCY (THE "VIBE" CHECK)
# ==========================================
# Simple cleaner to find the most used words
def get_top_words(text_series, top_n=20):
    all_text = ' '.join(text_series).lower()
    # Remove basic punctuation and numbers
    words = re.findall(r'\b[a-z]{3,}\b', all_text)
    
    # Common English stop words to ignore
    stop_words = {'the', 'and', 'this', 'that', 'for', 'you', 'with', 'are', 'was', 'but', 'have', 'they', 'not', 'just', 'like', 'from', 'what'}
    filtered_words = [w for w in words if w not in stop_words]
    
    return Counter(filtered_words).most_common(top_n)

In [13]:
top_words = get_top_words(df_comments['comment_text'])
df_words = pd.DataFrame(top_words, columns=['Word', 'Frequency'])

# Chart 3: Top Words
fig3 = px.bar(df_words, x='Word', y='Frequency', title="Top 20 Most Frequent Words", color='Frequency')
fig3.show()

print("\n🔍 EDA Complete. Review the charts above!")


🔍 EDA Complete. Review the charts above!


### Data Preparation

In [14]:
# ==========================================
# 2. THE SCRUBBER FUNCTION
# ==========================================
def clean_youtube_text(raw_text):
    if not isinstance(raw_text, str):
        return ""
    
    # Step A: Decode HTML entities (e.g., &quot; -> ", &amp; -> &)
    clean_text = html.unescape(raw_text)
    
    # Step B: Remove URLs (http://... or www....)
    clean_text = re.sub(r'http\S+|www.\S+', '', clean_text)
    
    # Step C: Remove excessive whitespace and newlines
    clean_text = re.sub(r'\s+', ' ', clean_text).strip()
    
    # Step D: Hard Truncation (AI Safety for RoBERTa 512-token limit)
    # 1500 characters is a very safe proxy for ~350 words.
    clean_text = clean_text[:1500] 
    
    return clean_text

In [15]:
# Load raw comments into Pandas
df = pd.read_sql("SELECT comment_id, comment_text FROM comments", engine)

In [16]:
df['cleaned_text'] = df['comment_text'].apply(clean_youtube_text)

# Find a comment that actually changed so we can see the difference
# (Looking for one where the length changed, meaning spaces/URLs/HTML were removed)
df['changed'] = df['comment_text'] != df['cleaned_text']
sample_changed = df[df['changed'] == True].head(1)

if not sample_changed.empty:
    print("\n--- BEFORE & AFTER VALIDATION ---")
    print(f"RAW:   {sample_changed.iloc[0]['comment_text']}")
    print(f"CLEAN: {sample_changed.iloc[0]['cleaned_text']}")
    print("------------------------------------\n")
else:
    print("\n Data was already remarkably clean! No major changes detected.")


--- BEFORE & AFTER VALIDATION ---
RAW:   I have lost all interest in Apple in the Cook era. Everything feels like a bunt, not a grand slam. The prices are insane, $1000 for a monitor stand and $500 for wheels to go on your Mac CPU. A big, shitty, shiny cash grab. 
That Neo laptop confused me when announced, because for once Apple wasn't aggressively ripping us off.
CLEAN: I have lost all interest in Apple in the Cook era. Everything feels like a bunt, not a grand slam. The prices are insane, $1000 for a monitor stand and $500 for wheels to go on your Mac CPU. A big, shitty, shiny cash grab. That Neo laptop confused me when announced, because for once Apple wasn't aggressively ripping us off.
------------------------------------



In [17]:
# Update the database securely
with engine.begin() as conn:
    for index, row in df.iterrows():
        # Using raw SQL to update the text in place
        update_query = text("""
            UPDATE comments 
            SET comment_text = :clean_val 
            WHERE comment_id = :id_val
        """)
        conn.execute(update_query, {"clean_val": row['cleaned_text'], "id_val": row['comment_id']})

print("Data Preparation Complete! Cleaned text saved to PostgreSQL.")

Data Preparation Complete! Cleaned text saved to PostgreSQL.


## API Testing

In [18]:
# Check if CUDA (NVIDIA GPU) is available
device_id = 0 if torch.cuda.is_available() else -1

if device_id == 0:
    print(f"GPU Detected: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected. Falling back to CPU (This will be slow).")

GPU Detected: NVIDIA GeForce GTX 1650


In [19]:
# Load the RoBERTa model fine-tuned for social media
MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"
sentiment_analyzer = pipeline(
    "sentiment-analysis", 
    model=MODEL_NAME, 
    tokenizer=MODEL_NAME, 
    device=device_id,
    max_length=512, # Strict transformer limit
    truncation=True
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 11175.62it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [20]:
# Label Mapping (Translating the model's raw output to clean text)
label_map = {
    'positive': 'Positive',
    'neutral': 'Neutral',
    'negative': 'Negative'
}

In [21]:
# ==========================================
# 3. RUN INFERENCE ON THE DATABASE
# ==========================================
print("Fetching cleaned comments from PostgreSQL...")
df = pd.read_sql("SELECT comment_id, comment_text FROM comments", engine)

total_comments = len(df)
print(f"Starting Inference on {total_comments} comments...")

# Create lists to hold our new data
new_labels = []
new_scores = []

# Process each comment
for index, row in df.iterrows():
    text_to_analyze = str(row['comment_text'])
    
    # If the comment is completely empty after cleaning, default to Neutral
    if not text_to_analyze.strip():
        new_labels.append('Neutral')
        new_scores.append(0.0)
        continue
        
    try:
        # The AI reads the text
        result = sentiment_analyzer(text_to_analyze)[0]
        
        # Save the results
        raw_label = result['label'].lower()
        new_labels.append(label_map.get(raw_label, 'Neutral'))
        new_scores.append(round(result['score'], 4))
        
    except Exception as e:
        print(f"Error on comment {row['comment_id']}: {e}")
        new_labels.append('Neutral')
        new_scores.append(0.0)

# Add the results back to our DataFrame
df['sentiment_label'] = new_labels
df['sentiment_score'] = new_scores

Fetching cleaned comments from PostgreSQL...
Starting Inference on 200 comments...


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [22]:
# ==========================================
# 4. SAVE RESULTS TO DATABASE
# ==========================================
print("\nUpdating Database with AI Insights...")

with engine.begin() as conn:
    for index, row in df.iterrows():
        update_query = text("""
            UPDATE comments 
            SET sentiment_label = :label_val, sentiment_score = :score_val 
            WHERE comment_id = :id_val
        """)
        conn.execute(update_query, {
            "label_val": row['sentiment_label'], 
            "score_val": row['sentiment_score'], 
            "id_val": row['comment_id']
        })

print("AI Inference Complete! Your database is now populated with sentiment data.")


Updating Database with AI Insights...
AI Inference Complete! Your database is now populated with sentiment data.
